# Browser Picker Pro — Technical Memory

- **File:** technical_memory.ipynb
- **Description:** Running log of *why* technical decisions were made in Browser Picker Pro — architecture choices, rejected alternatives, gotchas and wrong assumptions. Not a changelog (see `CHANGELOG.md`) and not a bug log (see `docs/BUGS.md`).
- **Author:** Jose-Jorge HERNANDEZ
- **Company:** Parlee Conseiller, Inc.
- **Date:** 2026-11-25
- **Last edit date:** 2026-11-26
- **Version:** 1.2.0

Entries are appended with a date heading; earlier entries are never rewritten.

## 2026-11-25 — v2.0: resident tray app + native launcher

### Context
v1.x was a one-shot process: every link started the 147 MB self-contained single-file exe, which booted .NET + WPF, built the DI container, read settings/rules, **scanned the disk and registry for browsers**, and only then showed the picker. After a pick it called `Shutdown()`, so every link paid the whole cost again — and the background video (the point of this eye-candy app) had to be opened and decoded from scratch each time.

### Decision
1. **One resident process** (named mutex `Local\BrowserPickerPro.Resident.<user>`) lives in the tray. The picker window is created once, *pre-warmed* (shown off-screen once so its HWND, visual tree and `MediaElement` pipeline exist), and afterwards only shown/hidden.
2. **Windows' link handler is a tiny NativeAOT exe** (`BrowserPickerProLauncher.exe`). It checks the mutex, calls `AllowSetForegroundWindow(ASFW_ANY)`, writes one line to a per-user named pipe and exits. No .NET runtime boot, no WPF.
3. If no resident instance exists, the launcher starts `BrowserPickerPro.exe` with the URL (cold path, same cost as v1).

### Alternatives rejected
- **Main exe does the forwarding itself** — works (kept as fallback when the main exe is invoked directly), but still pays .NET single-file extraction/boot before it can forward. Measured numbers are in `artifacts/picker_latency.ipynb`.
- **Trimming the working set on hide (`EmptyWorkingSet`)** — saves RAM but pages out the video decoder and the visual tree, so the next show page-faults them back in. Speed of the video background is the product, so RAM is deliberately traded for latency.
- **Keeping the video playing while hidden** — instant, but burns GPU/CPU forever. Pausing on hide and calling `Play()` on show resumes in a frame because the decoder stays open.
- **WinForms `NotifyIcon`** — needs `UseWindowsForms`, which drags in global-using ambiguities (`Application`, `Button`…) and an unthemed `ContextMenuStrip`. `H.NotifyIcon.Wpf` gives a WPF `ContextMenu` with our theme.

### Gotchas
- **Foreground lock.** A background process cannot steal focus; `Activate()` from the resident app just flashes the taskbar button. The launcher was started by the user's click so it owns the foreground right and passes it on with `AllowSetForegroundWindow(ASFW_ANY)` before writing to the pipe.
- **`RenderTransform` on a `Window` is forbidden** (CLAUDE.md gotcha #3), so the opening animations act on `MainContainer`/`InnerContainer`, never on the window.
- **A `Border`'s `Effect` slot holds one effect.** `MainContainer` carries the drop shadow, so the blur of the *zoom-blur* animation goes on `InnerContainer`.
- **Re-triggered entrance storyboard flash.** A `DataTrigger` without `ExitActions` keeps holding its final value (opacity 1) after it goes false, so re-showing the cards flashed them before the stagger restarted. Fixed with a named `BeginStoryboard` + `RemoveStoryboard` exit action.
- **PowerShell `$null` → `""`** when passed to a `string` P/Invoke parameter: `FindWindowW($null, title)` matched nothing. Use `[NullString]::Value`.

## 2026-11-25 (later) — what verification changed

### 1. Park the window instead of Hide()/Show()
The Debug breakdown showed ~55 ms of a ~100 ms reveal inside `Show()` of the layered (`AllowsTransparency`) window, even though it was pre-built. Decision: the resident picker stays **shown**, parked at (-32000,-32000) with content opacity 0, as a **tool window** (`WS_EX_TOOLWINDOW`, so not in Alt+Tab/taskbar). Reveal = move onto the monitor under the cursor. Consequences handled:
- `IsVisible` is now always true → code uses `IsRevealed`.
- A parked window could keep the keyboard focus → on *cancel* the previous foreground window (captured at reveal) is re-activated; after a *pick* it isn't, because the launched browser takes the focus.

### 2. Per-link work must be idempotent
`ApplySettings` ran fully on every link and re-created `ItemsPanel` → WPF regenerated all card containers. Same with `FilteredBrowsers.Clear()+Add()`. Both now only act on change.

### 3. Pre-warm must not wait for the video
The first version awaited `MediaOpened` (≤3 s) before declaring the picker ready. Harmless when warm, but on a cold start the first link waited on it (3.4 s first frame vs 1.4 s after the fix). `MediaOpened` already pauses the video when parked, so the wait was pointless.

### 4. The entrance-storyboard fix in the first entry was incomplete
`RemoveStoryboard` fixed the flash, but creating cards already `IsEntranceVisible=true` made the DataTrigger fire *during template instantiation*, where a template-local frozen `TranslateTransform` can't be animated ("Cannot animate '(0).(1)' on an immutable object instance"). Fix: transform moved to a Style **setter** (WPF clones setter values before animating) and cards are always created hidden.

### 5. Never swallow a render-loop exception forever
`DispatcherUnhandledException` → `Handled=true` is right for a one-off bad link, but a layout exception recurs every frame; it wrote 1 GB of log in minutes (Serilog's default per-file cap). Now: >5 errors in 10 s → let the process die (the launcher cold-starts it on the next link), and the file sink is capped at 10 MB with rolling.

### 6. Sync-over-async deadlock
`IsRegisteredAsync().GetAwaiter().GetResult()` on the UI thread hung startup: the method's `await Task.Run(...)` continuation was queued to the blocked dispatcher. Awaited now. Rule for this codebase: no `.Result`/`.GetResult()` on the UI thread, ever.

### Rejected
- **Trimming the working set when parked** — still rejected (see first entry); measured RSS 241–246 MB with the MP4 open is the price of an instant video.
- **Screenshots for timing** — a full-screen GDI capture stalls DWM ~150 ms, so frames land late; timing comes from the app log instead.

Related code (C#, from `BrowserPickerWindow.xaml.cs` and `App.xaml.cs`):


In [ ]:
%%script echo
// Parking (BrowserPickerWindow.Conceal → Park): the window stays shown, just far away.
private void Park()
{
    var hwnd = new WindowInteropHelper(this).Handle;
    if (hwnd != IntPtr.Zero) NativeMethods.MoveWindow(hwnd, OffscreenPosition, OffscreenPosition);
}

// Render-loop guard (App.OnDispatcherUnhandledException)
if (_recentUiErrors.Count > MaxUiErrorsPerWindow)   // > 5 within 10 s
{
    _logger?.LogCritical(e.Exception, "Repeated UI exceptions — exiting instead of looping");
    return;                                         // not handled → process ends
}
e.Handled = true;

## 2026-11-26 — The resident app must never run elevated (v2.1.0)

**Context.** Report: with Browser Picker Pro set as the default browser, opening a link or an `.html` file did *nothing at all*. There was no picker, no log line and no crash event.

**What misled the diagnosis.**
- Every test from Claude's shell worked (`Start-Process https://…`, calling the launcher directly). That shell is **elevated**, and so was the resident app: the previous session had started it from there after publishing.
- Today's log was listed as 0 bytes. That looked like broken logging, but NTFS updates the size in a directory listing lazily while the file is still open. The real log showed **no link at all** reached the app between 00:13 and 01:22.

**Root cause.** Integrity levels. Resident app = High (0x3000); Explorer and every normal app = Medium (0x2000). A Medium client can't open the named pipe of a High server, so `IpcProtocol.TrySend` returned false. The launcher then cold-started `BrowserPickerPro.exe`, which saw the resident mutex, failed to forward the URL the same way, and exited with code 0 without a word. Reproduced with `explorer.exe "https://…"` (runs the handler through the Medium desktop Explorer).

**Decision: de-elevate instead of opening the pipe up.** Lowering the pipe's mandatory label would have made the pipe reachable. But browsers launched by an elevated picker would then run as admin. Also, an elevated admin token's default owner is `BUILTIN\Administrators`, which can trip `PipeOptions.CurrentUserOnly` owner checks. Running the resident app at the user's normal level is the correct state, not a workaround.

**How.** First tried `CreateProcess` with `PROC_THREAD_ATTRIBUTE_PARENT_PROCESS` = Explorer. It failed with **ERROR_ELEVATION_REQUIRED (740)** even for an `asInvoker` exe with no compatibility shim. Switched to Raymond Chen's documented method: `ShellWindows.FindWindowSW(CSIDL_DESKTOP)` → `IServiceProvider.QueryService(SID_STopLevelBrowser)` → `IShellBrowser.QueryActiveShellView` → `IShellView.GetItemObject(SVGIO_BACKGROUND)` → `.Application.ShellExecute(exe, args)`. The child starts *from* Explorer with Explorer's token. Verified: started from the admin shell → resident at 0x2000, parent `explorer`.

**Guards.**
- Relaunch only when no resident instance exists. An elevated *forwarder* works (High → Medium is allowed).
- When UAC is off, Explorer itself is elevated, and relaunching through it would loop forever. Detect that via `TokenElevation` of the shell process and stay put with a warning.
- If a second instance still can't reach the resident app, it shows a one-shot picker instead of exiting silently. That fallback path was **not** exercised hands-on (it needs an unreachable resident app, which the fix now prevents).

In [ ]:
# Core of the decision in App.OnStartup (src/BrowserPickerPro.UI/App.xaml.cs), C#:
#
#   if (!IpcProtocol.IsResidentRunning() && Elevation.IsCurrentProcessElevated())
#   {
#       if (Environment.ProcessPath is { } exe &&
#           Elevation.TryRelaunchUnelevated(exe, e.Args, out deElevationFailure))
#       { Shutdown(); return; }
#   }
#   ...
#   if (!isFirstInstance)
#   {
#       if (ForwardToResident(args, started) || args.Url is not { Length: > 0 }) { Shutdown(); return; }
#       oneShotFallback = true;   // resident unreachable: show a picker ourselves
#   }
#
# Quick integrity check of the running resident app (PowerShell, admin shell):
#   explorer.exe "https://example.net/test"   # simulates a real, non-elevated click
